# CrossFuse v5 -- Stage C: Calibration + Cross-Dataset Evaluation

Fits calibration (temperature + bootstrap threshold) on VAL under the SAME
MC-Dropout estimator used at test time, reports the FakeAVCeleb test set,
then runs the honest cross-dataset suite:

- **DFDC sample, Celeb-DF v2** -- VIDEO HEAD ONLY. The fusion head
  is never OR-combined here: DFDC has no audio-only fakes, so adding the
  audio head can only inject false positives (this is exactly what dropped
  v3.2's accuracy from 60.4% to 55.2%). Fusion is reported on FakeAVCeleb only.
- **ASVspoof 2019 LA + In-the-Wild** -- audio head zero-shot, to separate
  generator-shift failure (ASVspoof, studio speech) from the domain-shift
  confound (In-the-Wild, YouTube speech, removes that confound).

Every number below carries n and a bootstrap 95% CI -- state both when
quoting a result.

Per-head confusion/ROC/reliability panels and the 4-way modality-attribution
matrix are saved as PNG to `/kaggle/working/figures/` -- after Save Version,
download them from this notebook's Output tab.

**Attach:** `crossfuse-v5-lib`, `crops-v5`, `crossfuse-v5-ckpt` (the `crossfuse_v5_main_*.pth` from `B-train`), a Celeb-DF v2 dataset and the DFDC sample (with `metadata.json`); optionally ASVspoof 2019 LA / In-the-Wild. Internet ON, GPU. Do not rely on `ffpp-crops-v5` here. FF++ held-out performance is measured in `B1`, not in this notebook.

**Status (2026-09): ran.** DFDC 0.8649 [0.8239, 0.9012], Celeb-DF 0.8447 [0.7813, 0.9025] (paper baseline 0.5799 / 0.6399). Outputs live in `results/`.

In [ ]:
# CLIP backbone needs open_clip. Installed --no-deps so pip cannot disturb Kaggle's
# torch/torchvision pairing (see B-train / B1 cell 1 for the dependency reasoning).
!pip install -q --no-deps open_clip_torch timm safetensors ftfy regex pyyaml
!pip install -q --no-deps facenet-pytorch
import sys, glob, os
_lib_hits = glob.glob("/kaggle/input/**/crossfuse_v5.py", recursive=True)
assert _lib_hits, ("crossfuse_v5.py not found under /kaggle/input -- attach the "
                   "crossfuse-v5-lib dataset in Add Input, then restart the session.")
sys.path.insert(0, os.path.dirname(_lib_hits[0]))
print("Using crossfuse_v5 library from:", _lib_hits[0])

import os, glob, csv, json
import numpy as np
import torch
import matplotlib.pyplot as plt

from crossfuse_v5 import (
    CONFIG, build_model, load_visual_encoder, mc_collect, fit_temperature,
    bootstrap_stable_threshold, expected_calibration_error, bootstrap_ci,
    classification_report_text,
    safe_auc, build_detector, extract_face_crops_contiguous, make_rng,
    run_training_pipeline,
)

# Pick the FakeAVCeleb manifest by its header, not by glob order: if ffpp-crops-v5
# is also attached, the first manifest.csv found could be the FF++ one.
_manifest_hits = []
for _p in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
    with open(_p) as _f:
        if "audio_label" in _f.readline():
            _manifest_hits.append(_p)
assert _manifest_hits, ("FakeAVCeleb manifest.csv (has an audio_label column) not "
                        "found under /kaggle/input -- attach the crops-v5 dataset "
                        "in Add Input, then restart the session.")
CROP_DIR = os.path.dirname(_manifest_hits[0])
with open(os.path.join(CROP_DIR, "manifest.csv")) as f:
    MANIFEST = list(csv.DictReader(f))
for r in MANIFEST:
    r["video_label"] = int(r["video_label"]); r["audio_label"] = int(r["audio_label"])
    r["fps"] = float(r["fps"])

_ckpt_hits = glob.glob("/kaggle/input/**/crossfuse_v5_main_*.pth", recursive=True)
assert _ckpt_hits, ("crossfuse_v5_main_*.pth not found under /kaggle/input -- attach the "
                    "crossfuse-v5-ckpt dataset in Add Input, then restart the session.")
CKPT_PATH = _ckpt_hits[0]
print(f"Loading checkpoint: {CKPT_PATH}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(CONFIG).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=False))
model.eval()

# Rebuild the SAME identity-disjoint split the checkpoint was trained under,
# so val/test here are the exact partitions calibration and reporting expect.
from crossfuse_v5 import build_3way_split_packed, CrossFuseCropDataset
from torch.utils.data import DataLoader
train_idx, val_idx, test_idx = build_3way_split_packed(
    MANIFEST, identity_disjoint=CONFIG["IDENTITY_DISJOINT_SPLIT"],
    fracs=CONFIG["SPLIT_FRACS"], seed=CONFIG["SEED"])
val_ds = CrossFuseCropDataset(MANIFEST, val_idx, CROP_DIR, CONFIG, training=False)
test_ds = CrossFuseCropDataset(MANIFEST, test_idx, CROP_DIR, CONFIG, training=False)
val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_CLIPS"], shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=CONFIG["BATCH_CLIPS"], shuffle=False, num_workers=2)


In [ ]:
# ---- Calibration fit on VAL, matched estimator ----
val_mc = mc_collect(model, val_loader, CONFIG, n_samples=CONFIG["N_MC_SAMPLES"])
temps, thrs = {}, {}
for head in ("video", "audio", "fusion", "sync"):
    if head not in val_mc:
        continue
    temps[head] = fit_temperature(val_mc[head]["mean_logit"], val_mc[head]["true"])
    probs = 1 / (1 + np.exp(-val_mc[head]["mean_logit"] / temps[head]))
    thrs[head] = bootstrap_stable_threshold(val_mc[head]["true"], probs)
    print(f"  {head:6s} thr={thrs[head]:.4f}  T={temps[head]:.2f}")

sync_reportable = bool(
    val_mc.get("sync", {}).get("true") is not None and
    safe_auc(val_mc["sync"]["true"], val_mc["sync"]["mean_logit"]) >= CONFIG["SYNC_MIN_AUC"])
calibration = {"temp": temps, "thr": thrs, "sync_reportable": sync_reportable}
with open("/kaggle/working/calibration_v5.json", "w") as f:
    json.dump({"temp": temps, "thr": thrs, "sync_reportable": calibration["sync_reportable"]},
              f, indent=2)


In [ ]:
# ---- Full report on TEST, matched estimator, bootstrap CIs ----
test_mc = mc_collect(model, test_loader, CONFIG, n_samples=CONFIG["N_MC_SAMPLES"])

print("=" * 74)
print("  CROSSFUSE v5 TEST-SET REPORT (calibrated, thresholds fit on VAL)")
print("=" * 74)
for head, label in (("video", "VIDEO HEAD"), ("audio", "AUDIO HEAD"),
                    ("fusion", "FUSION HEAD"), ("sync", "SYNC HEAD")):
    if head not in test_mc:
        continue
    d = test_mc[head]
    probs = 1 / (1 + np.exp(-d["mean_logit"] / temps.get(head, 1.0)))
    pred = (probs >= thrs.get(head, 0.5)).astype(int)
    print(f"\n--- {label} ---")
    print(classification_report_text(d["true"], pred,
                                     target_names=["Real/Matched", "Fake/Mismatched"]))
    auc = safe_auc(d["true"], probs)
    lo, hi = bootstrap_ci(d["true"], probs, safe_auc)
    ece = expected_calibration_error(probs, d["true"])
    print(f"AUC (n={len(d['true'])}): {auc:.4f} [{lo:.4f}, {hi:.4f}]  |  ECE: {ece:.4f}")
    if head == "sync" and auc < CONFIG["SYNC_MIN_AUC"]:
        print(f"[!] Sync AUC {auc:.4f} < preregistered SYNC_MIN_AUC {CONFIG['SYNC_MIN_AUC']} "
              "-- reported as a negative result, not a verdict signal. See paper Sec. V.")


In [ ]:
# ---- Figures for the paper: per-head confusion/ROC/reliability, and the
# 4-way modality-attribution matrix. Saved to /kaggle/working/figures/ as PNG
# -- download them from this notebook's Output tab after Save Version.
import matplotlib.pyplot as plt
from crossfuse_v5 import roc_curve as _roc_curve

FIG_DIR = "/kaggle/working/figures"
os.makedirs(FIG_DIR, exist_ok=True)

for head, label in (("video", "Video"), ("audio", "Audio"),
                    ("fusion", "Fusion"), ("sync", "Sync")):
    if head not in test_mc:
        continue
    d = test_mc[head]
    probs = 1 / (1 + np.exp(-d["mean_logit"] / temps.get(head, 1.0)))
    y_true = np.asarray(d["true"]).astype(int)
    thr = thrs.get(head, 0.5)
    pred = (probs >= thr).astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(y_true, pred):
        cm[t, p] += 1
    axes[0].imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
    axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Pred real", "Pred fake"])
    axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Actual real", "Actual fake"])
    axes[0].set_title(f"{label} -- Confusion (p>={thr:.2f})")

    fpr, tpr, _ = _roc_curve(y_true, probs)
    auc_val = safe_auc(y_true, probs)
    axes[1].plot(fpr, tpr, label=f"{label} (AUC={auc_val:.3f})")
    axes[1].plot([0, 1], [0, 1], "--", color="gray")
    axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
    axes[1].set_title(f"{label} -- ROC (test)"); axes[1].legend()

    edges = np.linspace(0, 1, 11)
    centers, accs = [], []
    for b in range(10):
        m = (probs > edges[b]) & (probs <= edges[b + 1])
        if m.sum() == 0:
            continue
        centers.append(probs[m].mean())
        accs.append((pred[m] == y_true[m]).mean())
    ece_val = expected_calibration_error(probs, y_true)
    axes[2].plot([0, 1], [0, 1], "--", color="black", label="Perfect calibration")
    axes[2].plot(centers, accs, "o-", color="red", label=f"{label} (ECE={ece_val:.3f})")
    axes[2].set_xlabel("Mean predicted confidence"); axes[2].set_ylabel("Empirical accuracy")
    axes[2].set_title(f"{label} -- Reliability (test)"); axes[2].legend()

    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f"{head}_eval_panel.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fig_path}")

# ---- 4-way modality-attribution confusion matrix ----
categories_test = [MANIFEST[i]["category"] for i in test_idx]
v_pred_all = (1 / (1 + np.exp(-test_mc["video"]["mean_logit"] / temps.get("video", 1.0)))
             >= thrs.get("video", 0.5)).astype(int)
a_pred_all = (1 / (1 + np.exp(-test_mc["audio"]["mean_logit"] / temps.get("audio", 1.0)))
             >= thrs.get("audio", 0.5)).astype(int)

CATS_ORDER = ["RealVideo-RealAudio", "RealVideo-FakeAudio",
             "FakeVideo-RealAudio", "FakeVideo-FakeAudio"]
SHORT_LABELS = ["RVRA\n(real/real)", "RVFA\n(real/fakeAud)",
               "FVRA\n(fakeVid/real)", "FVFA\n(fake/fake)"]
cat_idx = {c: i for i, c in enumerate(CATS_ORDER)}

conf4 = np.zeros((4, 4), dtype=int)
skipped = 0
for true_cat, vp, ap in zip(categories_test, v_pred_all, a_pred_all):
    if true_cat not in cat_idx:
        skipped += 1
        continue
    pred_cat = ("FakeVideo" if vp else "RealVideo") + "-" + ("FakeAudio" if ap else "RealAudio")
    conf4[cat_idx[true_cat], cat_idx[pred_cat]] += 1

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(conf4, cmap="Greens")
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(conf4[i, j]), ha="center", va="center",
               color="white" if conf4[i, j] > conf4.max() * 0.6 else "black", fontsize=12)
ax.set_xticks(range(4)); ax.set_xticklabels(SHORT_LABELS, fontsize=8)
ax.set_yticks(range(4)); ax.set_yticklabels(SHORT_LABELS, fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Modality Attribution -- Derived 4-Category Confusion (test)")
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "modality_attribution_4way.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path} ({skipped} test rows had an unrecognized category, excluded)")


In [ ]:
# ---- Cross-dataset generalization: DFDC sample, Celeb-DF v2 ----
# VIDEO HEAD ONLY on every one of these -- see the markdown header for why.
import json as _json

def score_videos_video_head(entries, desc, n_frames=32):
    """Video-level P(fake), averaged over EVERY window of the extracted run.

    v5 scored each video on the FIRST WINDOW_FRAMES (12) frames of a randomly
    placed 32-frame run, discarding ~60% of the frames it had already paid to
    extract and adding avoidable variance to every reported CI. This tiles the
    run into consecutive windows and averages their probabilities, which is
    also what the FF++ stage does (ffpp_v5.score_video_frames).
    """
    detector = build_detector(str(device))
    y_true, y_score = [], []
    from crossfuse_v5 import mc_logits
    W = CONFIG["WINDOW_FRAMES"]
    dummy_mfcc = torch.zeros(1, 40, CONFIG["AUDIO_FRAMES"]).to(device)
    for vpath, label in entries:
        crops, valid, *_ = extract_face_crops_contiguous(
            vpath, detector, rng=make_rng(vpath), n_frames=n_frames,
            out_size=CONFIG["CROP_SIZE"])
        if crops is None or int(valid.sum()) == 0:
            continue
        probs = []
        for s in range(0, max(1, crops.shape[0] - W + 1), W):
            win, vm = crops[s:s + W], valid[s:s + W]
            if win.shape[0] < W:
                pad = W - win.shape[0]
                win = np.concatenate([win, np.zeros((pad, *win.shape[1:]), np.uint8)])
                vm = np.concatenate([vm, np.zeros(pad, bool)])
            if not vm.any():
                continue          # no detected face in this window
            x = torch.from_numpy(np.ascontiguousarray(win)).unsqueeze(0).to(device)
            m = torch.from_numpy(vm).unsqueeze(0).to(device)
            got = mc_logits(model, x, dummy_mfcc, None, m,
                            n_samples=CONFIG["N_MC_SAMPLES"])
            probs.append(1 / (1 + np.exp(-got["video"].mean() / temps.get("video", 1.0))))
        if not probs:
            continue
        y_true.append(label)
        y_score.append(float(np.mean(probs)))
    return y_true, y_score
def report_video_generalization(name, y_true, y_score):
    if len(set(y_true)) < 2:
        print(f"[{name}] skipped -- fewer than 2 classes available (n={len(y_true)})")
        return None
    auc = safe_auc(y_true, y_score)
    lo, hi = bootstrap_ci(y_true, y_score, safe_auc)
    thr = thrs.get("video", 0.5)
    pred = (np.asarray(y_score) >= thr).astype(int)
    acc = float((pred == np.asarray(y_true)).mean())
    print(f"[{name}] n={len(y_true)} | AUC {auc:.4f} [{lo:.4f},{hi:.4f}] | "
          f"acc@val-thr {acc:.4f}")
    return {"name": name, "n": len(y_true), "auc": auc, "auc_ci": [lo, hi], "acc": acc}


results = {}

# -- DFDC sample (NOT the full 470GB set -- see plan Sec. re: sample size) --
dfdc_entries = []
for meta_path in glob.glob("/kaggle/input/**/metadata.json", recursive=True):
    folder = os.path.dirname(meta_path)
    with open(meta_path) as f:
        data = _json.load(f)
    for vid_name, info in data.items():
        full_path = os.path.join(folder, vid_name)
        if os.path.exists(full_path) and info.get("label", "").upper() in ("REAL", "FAKE"):
            dfdc_entries.append((full_path, 0 if info["label"].upper() == "REAL" else 1))
if dfdc_entries:
    yt, ys = score_videos_video_head(dfdc_entries, "DFDC")
    results["dfdc"] = report_video_generalization(
        f"DFDC sample (n_full_pool={len(dfdc_entries)})", yt, ys)
else:
    print("[i] No DFDC metadata.json found -- attach the DFDC competition data.")

# -- Celeb-DF v2 --
celebdf_entries = []
for vpath in glob.glob("/kaggle/input/**/Celeb-real/**/*.mp4", recursive=True):
    celebdf_entries.append((vpath, 0))
for vpath in glob.glob("/kaggle/input/**/YouTube-real/**/*.mp4", recursive=True):
    celebdf_entries.append((vpath, 0))
for vpath in glob.glob("/kaggle/input/**/Celeb-synthesis/**/*.mp4", recursive=True):
    celebdf_entries.append((vpath, 1))
if celebdf_entries:
    rng = make_rng("celebdf_eval"); rng.shuffle(celebdf_entries)
    celebdf_entries = celebdf_entries[:400]
    yt, ys = score_videos_video_head(celebdf_entries, "Celeb-DF")
    results["celebdf"] = report_video_generalization("Celeb-DF v2", yt, ys)
else:
    print("[i] No Celeb-DF data found -- attach a Celeb-DF v2 dataset.")

if results:
    aucs = [r["auc"] for r in results.values() if r]
    print(f"\nMean cross-dataset video AUC over {list(results)}: {np.mean(aucs):.4f}")

with open("/kaggle/working/crossdataset_results_v5.json", "w") as f:
    json.dump(results, f, indent=2)


In [ ]:
# ---- Audio zero-shot: ASVspoof 2019 LA + In-the-Wild ----
from crossfuse_v5 import mfcc_from_waveform, ffmpeg_extract_wav

def predict_audio_only(mfcc_batch):
    dummy_crops = torch.zeros(mfcc_batch.shape[0], CONFIG["WINDOW_FRAMES"],
                              CONFIG["CROP_SIZE"], CONFIG["CROP_SIZE"], 3,
                              dtype=torch.uint8).to(device)
    from crossfuse_v5 import mc_logits
    got = mc_logits(model, dummy_crops, mfcc_batch.to(device), None,
                    n_samples=CONFIG["N_MC_SAMPLES"])
    return 1 / (1 + np.exp(-got["audio"].mean(axis=0) / temps.get("audio", 1.0)))


def score_audio_files(entries, batch=32):
    y_true, y_score = [], []
    for i in range(0, len(entries), batch):
        chunk = entries[i:i + batch]
        feats = []
        for path, _ in chunk:
            try:
                y, sr = ffmpeg_extract_wav(path)
                feats.append(mfcc_from_waveform(y, sr, lead_skip=CONFIG["AUDIO_LEAD_SKIP"]))
            except Exception:
                feats.append(np.zeros((40, CONFIG["AUDIO_FRAMES"]), np.float32))
        probs = predict_audio_only(torch.from_numpy(np.stack(feats)))
        y_score.extend(probs.tolist())
        y_true.extend([lbl for _, lbl in chunk])
    return y_true, y_score


# ASVspoof 2019 LA eval partition
asv_proto = next(iter(glob.glob("/kaggle/input/**/*.trl.txt", recursive=True)), None) or \
            next(iter(glob.glob("/kaggle/input/**/ASVspoof2019.LA.cm.eval.trl.txt",
                                recursive=True)), None)
asv_flac_dir = next(iter(d for d in glob.glob("/kaggle/input/**/flac", recursive=True)
                         if "eval" in d.lower()), None)
if asv_proto and asv_flac_dir:
    entries = []
    with open(asv_proto) as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 5:
                fname, label = parts[1], parts[-1]
                fpath = os.path.join(asv_flac_dir, fname + ".flac")
                if os.path.exists(fpath):
                    entries.append((fpath, 0 if label == "bonafide" else 1))
    rng = make_rng("asv_eval"); rng.shuffle(entries)
    entries = entries[:1000]
    yt, ys = score_audio_files(entries)
    results["asvspoof"] = report_video_generalization(
        f"ASVspoof 2019 LA (studio speech, generator-shift only)", yt, ys)
else:
    print("[i] ASVspoof 2019 LA not found -- attach an ASVspoof dataset to run this.")

# In-the-Wild (YouTube speech -- removes the studio/YouTube domain confound)
itw_entries = []
itw_meta = next(iter(glob.glob("/kaggle/input/**/meta.csv", recursive=True)), None)
if itw_meta:
    import csv as _csv
    with open(itw_meta) as f:
        for row in _csv.DictReader(f):
            fpath = os.path.join(os.path.dirname(itw_meta), row.get("file", ""))
            lbl = row.get("label", "").lower()
            if os.path.exists(fpath) and lbl in ("bona-fide", "spoof"):
                itw_entries.append((fpath, 0 if lbl == "bona-fide" else 1))
if itw_entries:
    rng = make_rng("itw_eval"); rng.shuffle(itw_entries)
    itw_entries = itw_entries[:1000]
    yt, ys = score_audio_files(itw_entries)
    results["in_the_wild"] = report_video_generalization("In-the-Wild (YouTube speech)", yt, ys)
else:
    print("[i] In-the-Wild dataset not found -- attach it to run this.")

with open("/kaggle/working/crossdataset_results_v5.json", "w") as f:
    json.dump(results, f, indent=2)
print("\ncrossdataset_results_v5.json updated.")
